# Red Convolucional - CIFAR-10

3 modelos distintos con CNN.
CIFAR-10 tiene 60,000 imagenes a color 32x32. Distribuidas en 10 categorias: avion, automovil, pajaro, gato, ciervo, perro, rana, caballo, barco y camion.

In [1]:
import torch
import torch.nn as nn #capas de redes neuronales 
from torch.optim import Adam, SGD
from torch.utils.data import DataLoader #Divide el dataset en mini-batches
import torchvision #CIFAR-10
import torchvision.transforms as transforms #Herramientas para preprocesar/convertir las imágenes
import matplotlib.pyplot as plt
from tqdm import tqdm

In [2]:
#Buscaremos si tenemos GPU NIVIDIA, de no ser asi utilizaremos el CPU de respaldo

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")


Usando dispositivo: cuda


## 1. Carga y preprocesamiento del Dataset

In [3]:
# Transformamos las imagenes a tensores y normalizamos
transform = transforms.Compose([
    transforms.ToTensor(), # imagen PIL a Pytorch, [0,1] 
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.2470, 0.2435, 0.2616]) #media real del dataset CIFAR-10 
])

# Entrenamos y Descarga
train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

# Creamos dataset de prueba
test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

print(f"Imagenes de entrenamiento: {len(train_dataset)}")
print(f"Imagenes de prueba: {len(test_dataset)}")

Files already downloaded and verified
Files already downloaded and verified
Imagenes de entrenamiento: 50000
Imagenes de prueba: 10000


## 2. DataLoaders - División en mini-barches

In [4]:
class_names = ['avion', 'automóvil', 'pájaro', 'gato', 'ciervo', 'perro', 'rana', 'caballo', 'barco', 'camión']

BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print(f"Batch size: {BATCH_SIZE}")
print(f"Batches en train: {len(train_loader)}")
print(f"Batches en test:  {len(test_loader)}")

Batch size: 64
Batches en train: 782
Batches en test:  157


## 3. Funciones de entrenamiento y evaluación


In [5]:
def get_batch_accuracy(output, y, N):
    pred = output.argmax(dim=1)
    correct = pred.eq(y).sum().item()
    return correct / N

In [ ]:
def train(model, train_loader, test_loader, criterion, optimizer, num_epochs):
    history = {
        'train_loss': [],
        'test_loss': [],
        'train_acc': [],
        'test_acc': []
    }

    for epoch in tqdm(range(num_epochs), desc="Entrenando", unit="época"):

        # Entrenamiento
        model.train()
        train_loss = 0.0 
        train_acc = 0.0

        for X_barch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad() #Limpiamos los gradientes
            outputs = model(X_batch) #Forward pass
            loss = criterion(outputs, y_batch) # Calculamos perdida
            loss. backward() # Cambiar cada peso para reducir el error
            optimizer.step() #Actualizamos pesos

            train_loss += loss.item() * X_batch.size(0)
            train_acc += get_batch_accuracy(outputs, y_batch, len(train_loader.dataset))

        # Evaluación
        model.eval()
        test_loss = 0.0
        test_acc = 0.0

        with torch.no_grad():
            for X_test, y_test in test_loader:
                X_test = X_test.to(device)
                y_test = y_test.to(device)

                test_outputs = model(X_test)
                test_loss += criterion(test_outputs, y_test).item() * X_test.size(0)
                test_acc += get_batch_accuracy(test_outputs, y_test, len(test_loader.dataset))

        #Guardamos la metricas
        history['train_loss'].append(train_loss / len(train_loader.dataset))
        history['test_loss'].append(test_loss / len(test_loader.dataset))
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)

    return history